# RootScope on Google Colab

Segment and classify every cell in a confocal root-tip cross-section.

RootScope segments with **Cellpose-SAM**, extracts shape / size / intensity /
tissue-layer features plus fine-tuned **DINOv2** embeddings, and predicts each
cell's type with an iterative **RandomForest + XGBoost + LightGBM** ensemble.

- Code: https://github.com/ct-tranchau/Rootscope
- Weights: https://huggingface.co/ct-tranchau/Rootscope

> **Prerequisite:** this notebook installs RootScope from GitHub `main`, and
> uses the staged pipeline API (`stage_segment` / `stage_features` /
> `stage_embed` / `stage_classify`). Commit and push the `webapp/` work to
> `main` before sharing this notebook, or step 3 will stop with a clear error.

---

### Before you start: turn on the GPU

**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

On a GPU one image takes a couple of minutes. On CPU the same image takes
15-30 minutes, so this step is not optional in practice.


In [ ]:
#@title 1. Check the GPU is on { display-mode: "form" }
import subprocess, sys
try:
    print(subprocess.check_output(["nvidia-smi",
        "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip())
except Exception:
    print("NO GPU DETECTED.")
    print("Runtime > Change runtime type > T4 GPU, then re-run this cell.")
print("Python", sys.version.split()[0])

## 2. Install RootScope

This pulls in cellpose, xgboost, lightgbm and pins scikit-learn to 1.7.2 (the
version the released models were pickled with — other versions can fail to
unpickle).

**Colab will very likely ask you to restart the session afterwards.** That is
expected: pip changes versions of numpy/scikit-learn that Colab preloaded.
Click *Restart session*, then carry on from step 3 — you do **not** need to
re-run the install.

In [ ]:
#@title 2. Install (~3-5 min, restart when prompted) { display-mode: "form" }
%pip install -q "git+https://github.com/ct-tranchau/Rootscope.git"
print("\nInstall finished.")
print("If Colab shows a RESTART SESSION button, click it, then continue at step 3.")

## 3. Load the models

First run downloads ~460 MB of classifiers from Hugging Face, the ~85 MB
fine-tuned DINOv2 backbone, and the Cellpose-SAM checkpoint. They are cached,
so later runs in the same session start instantly.

In [ ]:
#@title 3. Load models onto the GPU { display-mode: "form" }
import torch
from rootscope import predict as rs
from rootscope.cnn_embeddings import load_dinov2
from rootscope.extract_features import load_cellpose_model
from rootscope.weights import resolve_cnn_weights, resolve_model_dir

# This notebook drives the pipeline stage by stage so segmentation and the
# DINOv2 embeddings can run on the GPU while the rest runs on the CPU. Those
# stage_* helpers landed with the web-app work; older installs only have the
# all-in-one predict_tif().
missing = [n for n in ("stage_segment", "stage_features", "stage_embed",
                       "stage_classify", "load_image") if not hasattr(rs, n)]
if missing:
    raise RuntimeError(
        "The installed RootScope is too old for this notebook - missing: "
        + ", ".join(missing) + ".\n"
        "The staged API must be pushed to GitHub main before this notebook "
        "works. Until then, use the simple one-call API instead:\n\n"
        "    from rootscope import predict_tif\n"
        "    df = predict_tif(TIF, out_dir='results', gpu=True, um_per_px=UM_PER_PX)\n")

GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "NONE - this will be slow")

MODEL_DIR   = resolve_model_dir()
CNN_WEIGHTS = resolve_cnn_weights()
MODELS, SCALERS, FEATURE_COLS, LE = rs.load_models(str(MODEL_DIR))
CELLPOSE = load_cellpose_model(use_gpu=GPU)
DINOV2   = load_dinov2(weights_path=str(CNN_WEIGHTS) if CNN_WEIGHTS else None,
                       use_gpu=GPU)
print(f"\nReady: {len(MODELS)} classifiers, {len(FEATURE_COLS)} features")
print("Classes:", list(LE.classes_))

## 4. Upload your TIFF

### The pixel size matters

RootScope does **not** read the pixel scale out of the file, and the 1.0
default distorts every size-derived feature. The cell below reads
`PhysicalSizeX` from the OME metadata when it is there and tells you what it
found — check it, and override `UM_PER_PX` below if your file has no metadata.

For the bundled examples: `Acorulea` = 0.6478, `Spennellii` = 0.4546.

In [ ]:
#@title 4. Upload a TIFF (or skip to use an example) { display-mode: "form" }
import re, tifffile
from pathlib import Path
from google.colab import files

USE_EXAMPLE = True  #@param {type:"boolean"}

if USE_EXAMPLE:
    import urllib.request
    TIF = "Spennellii_RootTip_EarlyMaturation.tif"
    if not Path(TIF).exists():
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/ct-tranchau/Rootscope/main/"
            "examples/Spennellii_RootTip_EarlyMaturation.tif", TIF)
    print("Using bundled example:", TIF)
else:
    uploaded = files.upload()
    TIF = list(uploaded.keys())[0]
    print("Uploaded:", TIF)

def read_um_per_px(path):
    try:
        with tifffile.TiffFile(path) as tf:
            if tf.ome_metadata:
                m = re.search(r'PhysicalSizeX="([0-9.eE+-]+)"', tf.ome_metadata)
                if m:
                    return round(float(m.group(1)), 6)
    except Exception as e:
        print("  could not read metadata:", e)
    return None

detected = read_um_per_px(TIF)
if detected:
    UM_PER_PX = detected
    print(f"Pixel size from metadata: {UM_PER_PX} um/px")
else:
    UM_PER_PX = 1.0
    print("NO pixel size in metadata. Set UM_PER_PX by hand in the next cell,")
    print("otherwise every size-derived feature will be wrong.")

In [ ]:
#@title 4b. Override the pixel size if you need to { display-mode: "form" }
OVERRIDE = False   #@param {type:"boolean"}
VALUE    = 1.0     #@param {type:"number"}
if OVERRIDE:
    UM_PER_PX = VALUE
print("Using UM_PER_PX =", UM_PER_PX)

## 5. Run

Segmentation and the DINOv2 embeddings run on the GPU; feature extraction and
the iterative ensemble run on the CPU. All four models (RandomForest, XGBoost,
LightGBM, Ensemble) are produced in one pass — `Ensemble` is the published
result.

In [ ]:
#@title 5. Segment + classify { display-mode: "form" }
import time
from pathlib import Path

OUT = Path("results"); OUT.mkdir(exist_ok=True)
stem = Path(TIF).stem
t0 = time.time()

img = rs.load_image(TIF)
print("Image:", img.shape)

masks = rs.stage_segment(img, gpu=GPU, cellpose_model=CELLPOSE)
print(f"Segmented: {int(masks.max())} cells  ({time.time()-t0:.0f}s)")

masks, df_base, layers, adjacency, n_layers = rs.stage_features(
    masks, img, um_per_px=UM_PER_PX)
df_base = rs.stage_embed(masks, img, df_base, gpu=GPU, dinov2_model=DINOV2)

df = rs.stage_classify(
    df_base, masks, img, layers, adjacency,
    MODELS, SCALERS, FEATURE_COLS, LE,
    out_dir=OUT, stem=stem, source_name=Path(TIF).name,
    um_per_px=UM_PER_PX, max_rounds=10, label_cells=False)

print(f"\nDone in {time.time()-t0:.0f}s -> {OUT}/")

## 6. Look at the result

In [ ]:
#@title 6. Show the overlay and per-type counts { display-mode: "form" }
import matplotlib.pyplot as plt
from PIL import Image

MODEL = "Ensemble"  #@param ["Ensemble", "RandomForest", "LightGBM", "XGBoost"]

overlay = OUT / f"{stem}_{MODEL}_overlay.png"
plt.figure(figsize=(11, 11))
plt.imshow(Image.open(overlay)); plt.axis("off"); plt.title(f"{stem} - {MODEL}")
plt.show()

sub = df[df["model"] == MODEL]
counts = sub["predicted_cell_type"].value_counts()
summary = counts.to_frame("cells")
summary["% of cells"] = (100 * counts / counts.sum()).round(1)
summary["mean confidence"] = [
    round(sub.loc[sub["predicted_cell_type"] == ct, "prediction_confidence"].mean(), 3)
    for ct in counts.index]
print(f"{len(sub)} cells, {n_layers} tissue layers, {UM_PER_PX} um/px")
print(f"xylem poles: {int(sub.iloc[0]['n_xylem_poles'])}, "
      f"phloem poles: {int(sub.iloc[0]['n_phloem_poles'])}\n")
display(summary)

## 7. Download the results

In [ ]:
#@title 7. Download everything as a ZIP { display-mode: "form" }
import shutil
from google.colab import files
shutil.make_archive(f"{stem}_rootscope", "zip", OUT)
files.download(f"{stem}_rootscope.zip")

---

## Batch: a folder of images from Google Drive

Put your TIFFs in a Drive folder and point `TIF_DIR` at it. Results are written
back to Drive, so nothing is lost when the session ends.

One `UM_PER_PX` applies to the whole folder — group images by pixel size, or
run the folder once per scale.

In [ ]:
#@title Batch over a Drive folder { display-mode: "form" }
RUN_BATCH = False  #@param {type:"boolean"}

if RUN_BATCH:
    from google.colab import drive
    import pandas as pd
    drive.mount("/content/drive")

    TIF_DIR     = "/content/drive/MyDrive/root_tifs"      #@param {type:"string"}
    OUT_DIR     = "/content/drive/MyDrive/root_results"   #@param {type:"string"}
    BATCH_UM_PX = 0.4546                                  #@param {type:"number"}

    out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)
    tifs = sorted(Path(TIF_DIR).glob("*.tif"))
    print(f"{len(tifs)} images found\n")

    tables = []
    for i, tp in enumerate(tifs, 1):
        print(f"[{i}/{len(tifs)}] {tp.name}")
        try:
            im = rs.load_image(tp)
            mk = rs.stage_segment(im, gpu=GPU, cellpose_model=CELLPOSE)
            if int(mk.max()) == 0:
                print("   no cells, skipped"); continue
            mk, base, ll, adj, nl = rs.stage_features(mk, im, um_per_px=BATCH_UM_PX)
            if base is None:
                print("   all debris, skipped"); continue
            base = rs.stage_embed(mk, im, base, gpu=GPU, dinov2_model=DINOV2)
            d = rs.stage_classify(base, mk, im, ll, adj,
                                  MODELS, SCALERS, FEATURE_COLS, LE,
                                  out_dir=out, stem=tp.stem, source_name=tp.name,
                                  um_per_px=BATCH_UM_PX, max_rounds=10)
            tables.append(d)
        except Exception as e:
            print(f"   FAILED: {e}")

    if tables:
        combined = pd.concat(tables, ignore_index=True)
        combined.to_csv(out / "all_predictions.csv", index=False)
        print(f"\n{len(combined)} rows -> {out}/all_predictions.csv")

---

## Optional: a Gradio web UI with a public link

Runs the same interface as the Hugging Face Space, right here on Colab's GPU,
and prints a public `*.gradio.live` URL you can share.

The link lives as long as this cell runs, up to 72 hours. Colab also
disconnects idle sessions, so treat it as a demo link rather than hosting.

In [ ]:
#@title Launch the Gradio UI { display-mode: "form" }
LAUNCH_UI = False  #@param {type:"boolean"}

if LAUNCH_UI:
    %pip install -q gradio
    import urllib.request, urllib.error
    URL = ("https://raw.githubusercontent.com/ct-tranchau/Rootscope/main/"
           "webapp/app.py")
    try:
        urllib.request.urlretrieve(URL, "app.py")
    except urllib.error.HTTPError as e:
        raise SystemExit(
            f"Could not fetch app.py ({e.code}). The webapp/ folder has to be "
            "committed and pushed to GitHub main first.")
    # app.py loads its own models; the `spaces` import is optional so the
    # @spaces.GPU decorators become no-ops here and it just uses Colab's GPU.
    import app
    app.demo.queue(max_size=10).launch(share=True,
                                       allowed_paths=[str(app.TMP_ROOT)])

---

## Troubleshooting

| Problem | Fix |
|---|---|
| `NO GPU DETECTED` | Runtime → Change runtime type → T4 GPU, then re-run from step 1 |
| Colab asks to restart after install | Expected. Restart, then continue at step 3 — don't re-run the install |
| `numpy`/`sklearn` version errors | Restart the session. If it persists, Runtime → Disconnect and delete runtime, then start over |
| Cells look wrong / sizes implausible | `UM_PER_PX` is probably still 1.0. Set it in step 4b |
| Out of memory | Peak use is ~3.3 GB, well within Colab. If it happens, restart and avoid re-running step 3 twice |
| Session died mid-batch | Colab caps free sessions (~12 h, less when idle). Write results to Drive, as the batch cell does |

## Notes

- Predictions are for research use. See the
  [model card](https://github.com/ct-tranchau/Rootscope/blob/main/MODEL_CARD.md)
  for the species and growth stages the model was trained on.
- Z-stacks are max-projected before segmentation.
- Colab's free GPU is not guaranteed; at busy times you may be given a CPU
  runtime, where one image takes 15-30 minutes.
